# 모듈 1. LangChain 기초 — 입력 다루기

> 이 노트북에서는 LangChain의 기본 사용법을 학습합니다.  
> OpenAI API 직접 호출과 LangChain을 통한 호출을 비교하고, 메시지 처리(invoke/stream/batch), PromptTemplate, 대화 메모리, 캐싱, 비동기 처리를 실습합니다.

**학습 목표**
- LangChain을 통해 OpenAI 모델을 호출하는 방법 이해
- PromptTemplate / ChatPromptTemplate으로 동적 프롬프트 구성
- ChatMessageHistory로 멀티턴 대화 관리
- InMemoryCache로 API 비용 절감

In [ ]:
# langchain 관련 라이브러리 설치하기 (LangChain 1.0+)
# langchain 1.0에서는 core/community/provider 패키지가 분리됨
# pip install langchain langchain-core langchain-community langchain-openai
# pip install langchain langchain-core langchain-community langchain-openai

## 0.OpenAI 의 API 사용
| 항목                | Chat Completions             | Responses     |
| ----------------- | ---------------------------- | ------------- |
| 현재 권장 여부          | 기존 API                       | **권장 API**    |
| 입력                | `messages`                   | `input`       |
| 출력                | `choices[0].message.content` | `output_text` |
| 이미지 입력            | 가능(제한적)                      | 매우 편리         |
| 파일 입력             | 별도 방식                        | 통합 지원         |
| Tool Calling      | 가능                           | 더 개선됨         |
| Structured Output | 가능                           | 더 자연스러움       |
| 멀티모달              | 가능                           | **통합 설계**     |

- chat completions 방식은 대화 메시지 배열을 보내고 답변 하나를 받는 방식
- Respose 방식은 텍스트 생성 + 도구 호출 + 이미지/파일/웹검색/코드 실행 같은 작업까지 더 넓게 처리하는 통합 방식

- Response API 방식

In [2]:
# OpenAI 의 API 사용
from openai import OpenAI

# OpenAI 클라이언트 초기화
clinet = OpenAI()

# AI 요청
response = clinet.responses.create(
    model = "gpt-4o-mini",
    input = [
        {"role": "system", 
         "content": "당신은 서울의 음식과 문화 전문가입니다."},
        {"role": "user",
        "content": "서울을 대표하는 음식을 맛볼 수 있는 레스토랑 5개를 추천해 주세요."}
    ]
)

# 응답 내용 출력
print(response.output_text)

서울에서 대표적인 음식을 맛볼 수 있는 레스토랑을 아래와 같이 추천드립니다.

1. **광장시장 - 빈대떡과 마님떡볶이**
   - 광장시장은 다양한 길거리 음식을 즐길 수 있는 명소입니다. 빈대떡과 함께 마님떡볶이를 추천합니다. 현지의 얼큰하고 매콤한 떡볶이를 경험할 수 있어요.

2. **진주회관**
   - 갈비탕, 비빔밥으로 유명한 레스토랑으로, 다양한 한국 전통 음식을 맛볼 수 있습니다. 특히 갈비탕이 깊고 진한 국물이 특징입니다.

3. **부암동 - 정선유기**
   - 고급 한정식이나 전통 한식을 경험하고 싶다면 이곳을 추천합니다. 제철 재료로 만든 다양한 반찬과 고유의 한식을 즐길 수 있습니다.

4. **신사동 - 가츠라**
   - 일본식 돈까스 전문점이지만, 서울의 다양한 맛을 접할 수 있습니다. 특히 일본식 토마토 브랜디 소스가 일품입니다.

5. **동대문 - 칼국수집**
   - 딱 한 끼로 진한 국물 맛의 칼국수와 freshly made 만두를 즐길 수 있는 곳입니다. 쫄깃한 면과 바삭한 만두의 조합이 좋습니다.

서울의 정통 음식을 즐길 수 있는 이들 레스토랑을 방문하면 현지의 맛을 깊이 느낄 수 있습니다!


## 1. Hello LangChain
LangChain으로 LLM을 가장 간단하게 호출하는 법

OpenAI SDK로는 client.responses.create(model=..., input=[...])처럼 여러 인자를 넣어야 했는데, LangChain은 client.invoke("질문") 딱 한 줄로 끝남.

- LangChain 에서 Openai 의 API 사용 - 기본 chat

In [12]:
# LangChain 에서 OpenAI의 API 사용

import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import OpenAI

# .env 파일에서 환경변수 로드
load_dotenv(find_dotenv(), override=True)

# OpenAI 클라이언트 초기화
client = OpenAI()  

# AI 응답 요청 - invoke 메서드 사용. prompt 자유롭게 작성하기
response = client.invoke("서초구 맛집 5개를 추천해줘")  # 한 줄로 끝남. Respose API와 비교했을 때 훨씬 간단해짐.

# 응답 내용 출력
print(response)



1. 미미네 맛집
2. 스시키마
3. 모모카페
4. 루치끼
5. 봉우리식당


In [23]:
from langchain_openai import ChatOpenAI

client = ChatOpenAI(
    model = "gpt-4o-mini",
    max_tokens=200
)

response = client.invoke("서초구 맛집 3개 추천")
print(response.content)

서초구에는 다양한 맛집이 있습니다. 아래는 추천할 만한 3곳입니다.

1. **한일관**: 전통적인 한식 맛집으로, 특히 국물 요리와 다양한 찌개가 유명합니다. 고급스러운 분위기에서 정통 한국 음식을 즐길 수 있습니다.

2. **툴랄라**: 이탈리안 레스토랑으로, 파스타와 피자 등 다양한 메뉴를 제공합니다. 아늑한 분위기에서 진정한 이탈리안 요리를 맛볼 수 있는 곳입니다.

3. **명동교자**: 칼국수와 만두가 유명한 집입니다. 특제 육수와 쫄깃한 면이 일품입니다. 서초구에도 지점이 있어 부담 없이 간편하게 즐길 수 있습니다.

맛집을 방문하시기 전에 예약이나 운영 시간을 확인하시는 것이 좋습니다!


## 2. LangChain 메세지 처리



In [24]:
# langchain 을 사용하여 openai 모델

from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage, SystemMessage, HumanMessage
from dotenv import load_dotenv, find_dotenv
import os

# .env 파일에서 환경변수 로드
load_dotenv(find_dotenv(), override=True)

# OpenAI 클라이언트 초기화
cline = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature=0.7
)

In [30]:
messages = [
    SystemMessage(content="당신은 여행 전문가로 고객의 여행 일정에 도움을 줍니다."),
    HumanMessage(content="부산 여행에서 딱 한 곳만 가봐야 한다면 어떤 곳인지 알려줘.")
]

response = clinet.invoke(messages) # response는 이미 AIMessages가 됨.
messages.append(response)

for msg in messages:
    print(f"[{msg.__class__.__name__}] {msg.content}")

messages.append(HumanMessage(content="그곳에 가면 꼭 먹어야 할 음식도 알려주세요."))
response2 = client.invoke(messages)
messages.append(response2)

for msg in messages[-2:]:
    print(f"[{msg.__class__.__name__}] {msg.content}")

[SystemMessage] 당신은 여행 전문가로 고객의 여행 일정에 도움을 줍니다.
[HumanMessage] 부산 여행에서 딱 한 곳만 가봐야 한다면 어떤 곳인지 알려줘.
[AIMessage] 부산 여행에서 딱 한 곳만 가봐야 한다면 **해운대 해수욕장**을 추천합니다. 해운대는 부산에서 가장 유명한 해변으로, 아름다운 바다와 백사장, 그리고 다양한 먹거리와 즐길 거리가 가득합니다. 해변 산책은 물론, 근처의 동백섬과 해운대 마린시티의 멋진 경치를 즐길 수 있습니다. 또한, 여름철
[HumanMessage] 그곳에 가면 꼭 먹어야 할 음식도 알려주세요.
[AIMessage] 해운대 해수욕장을 방문하면 꼭 드셔야 할 음식은 **회국수**와 **해운대 씨앗호떡**입니다.

1. **회국수**: 부산의 신선한 회와 찬 국수를 함께 즐길 수 있는 음식입니다. 해산물의 신선함과 국수의 쫄깃함이 조화를 이루어 한 끼 식사로 적합합니다.

2. **해운대 씨앗호떡**: 해운대 지역의 특별한 간식으로, 안에 견과류와 설탕 시럽이 들어간 호떡입니다. 바삭하면서도 달콤한 맛이 일품이며, 해변에서 즐기기에 좋은 간식입니다.

이 두 가지 음식을 꼭 시도해보세요! 부산의 맛을 제대로 느낄 수 있을 거예요.


- **LLM은 Stateless(상태를 저장하지 않는 구조) 이기 때문에 이전 대화를 기억하지 못한다.**
- SystemMessage + HumanMessage + AIMessage + HumanMessage + AIMessage 방식으로 계속 추가해줘야 함

## 4.동기/비동기 처리


* async / await란?
    - async def → 비동기 함수 정의 (일반 함수 대신 비동기 실행 가능)
    - await → 기다릴 수 있는 작업 (주로 async 함수 안에서만 사용)

In [31]:
# 동기 방식 (느림)
import time

start = time.time()  # 시작 시간

def say_hello():
    time.sleep(2)  # 2초 기다림
    print("Hello")

def say_world():
    time.sleep(1)
    print("World")

say_hello()
say_world()

end = time.time()  # 종료 시간

print(f"\n총 실행 시간: {end - start:.2f}초")

Hello
World

총 실행 시간: 3.01초


In [32]:
import asyncio
import time

start = time.time()

async def say_hello():
    await asyncio.sleep(2)
    print("Hello")

async def say_world():
    await asyncio.sleep(1)
    print("World")

async def main():
    await asyncio.gather(
        say_hello(),
        say_world()
    )

await main()

end = time.time()
print(f"\n총 실행 시간: {end - start:.2f}초")


World
Hello

총 실행 시간: 2.03초


In [33]:
# 1. 비동기 처리와 시간 측정을 위해 모듈 임포트
import asyncio
import time

# 2. 비동기 요청 함수 정의
#    - client.ainvoke()를 사용하여 LLM에게 메시지를 전달하고 응답을 받음
async def invoke_async(client, messages):
    response = await client.ainvoke(messages)   # 비동기 호출
    print(response.content)                     # 응답 출력

# 3. 병렬 실행 함수 정의
#    - 같은 요청을 10번 병렬로 실행
async def invoke_parallel(client, messages):
    tasks = [invoke_async(client, messages) for _ in range(5)]  # 비동기 작업 리스트
    await asyncio.gather(*tasks)                                # 동시에 실행

# 4. 메시지 정의 (System + Human 역할)
messages = [
    ("system", "당신은 서울의 음식 전문가입니다."),
    ("human", "서울 광장시장에서 먹을 만한 길거리 음식들을 소개해 주세요")
]

# 5. 병렬 실행 시작 및 시간 측정 -------------------------------------
print("Async")  # 병렬 실행 여부 출력
start = time.perf_counter()  # 시작 시간 기록

# invoke_parallel 함수 실행 (5개의 요청 병렬 실행)
await invoke_parallel(client, messages)

end = time.perf_counter()  # 종료 시간 기록
print(f"Elapsed time: {end - start:.2f} seconds")  # 총 소요 시간 출력

# 6. 동기 방식 실행
print("Sync")  # 동기방식 출력
start = time.perf_counter()                  # 시작 시간 기록
for i in range(5):                           # 5번 반복 (직렬 실행)
    response = client.invoke(messages)       # 동기 호출
    print(response.content)                  # 응답 출력
    print()                                  # 줄바꿈
end = time.perf_counter()                    # 종료 시간 기록
print(f"Elapsed time: {end - start:.2f} seconds")  # 걸린 시간 출력


Async
서울 광장시장은 다양한 길거리 음식을 즐길 수 있는 명소입니다. 여기서 꼭 먹어봐야 할 음식들을 소개해드릴게요.

1. **빈대떡**: 전통적인 한국 전 전병으로, 찹쌀가루와 다양한 채소(파, 고추 등)를 섞어 구워냅니다. 바삭하면서도 부드러운 식감을 느낄 수 있습니다.

2. **떡볶이**: 매콤달콤한 소스에 쫄깃한 떡과 어묵이 어우러진 인기 간식입니다. 광장시장에서 파는 떡볶이는 특유의 매운 맛과 함께 깊은 맛을 느낄 수 있습니다.

3. **오뎅 (어묵)**: 각종 어묵을 꼬치에 꽂아 뜨끈한 육수와 함께 판매
서울 광장시장은 다양한 길거리 음식으로 유명한 곳입니다. 여기를 방문하면 꼭 먹어봐야 할 몇 가지 추천 음식을 소개할게요.

1. **빈대떡**: 뜨거운 기름에 바삭하게 튀겨내는 빈대떡은 고소한 맛이 일품입니다. 주로 김치나 해물 빈대떡으로 즐길 수 있습니다.

2. **떡볶이**: 매콤달콤한 소스에 쫄깃한 떡과 어묵이 어우러진 떡볶이는 대한민국 길거리 음식의 대표주자입니다. 다양한 토핑을 추가하여 취향대로 즐길 수 있습니다.

3. **순대**: 돼지 내장을 사용한 순대는 찰지고 고소한 맛이 특징입니다. 보통 양념장과 함께 제공되며, 삶은 순대와
서울 광장시장은 다양한 길거리 음식으로 유명합니다. 이곳에서 맛볼 수 있는 인기 있는 길거리 음식 몇 가지를 소개해 드릴게요.

1. **떡볶이**: 매콤하면서도 달콤한 소스에 볶은 떡, 어묵, 그리고 때로는 치킨이나 야채가 함께 제공됩니다. 광장시장에서의 떡볶이는 정말 인기가 많습니다.

2. **순대**: 한국 전통의 순대는 각종 내장과 채소, 당면으로 만들어지며, 간장 소스 또는 고추장과 함께 드시면 더욱 맛있습니다.

3. **호떡**: 달콤한 시럽과 견과류가 들어간 부드러운 팬케이크 형태의 간식으로, 뜨거운 와중에 먹으면 겉은 바삭하고 속은 달콤합니다.

4.
광장시장은 서울에서 가장 유명한 재래시장 중 하나로, 다양한 길거리 음식을 맛볼 수 있는 명소입니다. 여기 몇 가지 추천할 만한

In [35]:
# 실무 예시
import asyncio
import time
from langchain_openai import ChatOpenAI

clinet = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature = 0.3
)

# 개별 비동기 작업 함수
async def run_task(task_name: str, prompt: str):
    print(f"[시작] {task_name}")

    response = await clinet.ainvoke([
        ("system", "당신은 실무형 AI 업무 지원 전문가입니다."),
        ("human", prompt)
    ])

    print(f"\n[{task_name} 결과]")
    print(response.content)
    print(f"[완료] {task_name}")

    return response.content


# 여러 작업을 동시에 실행
async def main():
    tasks = [
        run_task(
            "회의록 요약",
            "다음 회의 내용을 5줄로 요약하고, 결정사항과 후속 작업을 구분해줘."
        ),
        run_task(
            "고객 문의 답변 초안",
            "배송 지연에 대해 사과하고 예상 배송일을 안내하는 고객 답변 메일 초안을 작성해줘."
        ),
        run_task(
            "보고서 목차 생성",
            "생성형 AI 기반 교육 프로그램 성과 보고서의 목차를 작성해줘."
        )
    ]

    results = await asyncio.gather(*tasks)
    return results

# 실행 및 시간 측정
start = time.perf_counter()
results = await main()

end = time.perf_counter()
print(f"\n총 소요 시간: {end - start:.2f}초")

[시작] 회의록 요약
[시작] 고객 문의 답변 초안
[시작] 보고서 목차 생성

[회의록 요약 결과]
회의 내용을 제공해 주시면 요약과 결정사항, 후속 작업을 구분해 드리겠습니다. 내용을 입력해 주세요!
[완료] 회의록 요약

[고객 문의 답변 초안 결과]
제목: 배송 지연에 대한 사과 및 예상 배송일 안내

안녕하세요, [고객님 성함]님.

먼저, 저희 제품을 주문해 주셔서 진심으로 감사드립니다. 그러나 배송 지연으로 인해 불편을 드리게 되어 진심으로 사과의 말씀을 드립니다.

현재 주문하신 상품은 예상보다 배송이 지연되고 있으며, 최종 배송일은 [예상 배송일]로 예정되어 있습니다. 저희는 고객님께서 기다리시는 동안 불편함이 없도록 최선을 다하고 있으며, 가능한 한 빠르게 상품을 배송할 수 있도록 노력하고 있습니다.

배송 진행 상황에 대한 추가적인 정보가 필요하시거나 궁금한 점이 있으시면 언제든지 저희 고객센터로 문의해 주시기 바랍니다. 고객님의 소중한 의견을 항상 귀 기울여 듣겠습니다.

다시 한번 불편을 드린 점 사과드리며, 빠른 시일 내에 상품을 받아보실 수 있도록 최선을 다하겠습니다.

감사합니다.

[귀하의 이름]  
[귀하의 직책]  
[회사명]  
[연락처]  
[이메일 주소]  
[완료] 고객 문의 답변 초안

[보고서 목차 생성 결과]
생성형 AI 기반 교육 프로그램 성과 보고서의 목차는 다음과 같이 구성할 수 있습니다:

1. **서론**
   - 1.1. 보고서 목적
   - 1.2. 교육 프로그램 개요
   - 1.3. 연구 배경 및 필요성

2. **프로그램 개요**
   - 2.1. 프로그램 목표
   - 2.2. 대상 및 참여자
   - 2.3. 교육 내용 및 커리큘럼
   - 2.4. 교육 방식 및 도구

3. **성과 평가 방법론**
   - 3.1. 평가 기준 및 지표
   - 3.2. 데이터 수집 방법
   - 3.3. 분석 방법

4. **성과 분석**
   - 4.1. 참여자 성과
       - 4.1.1. 학습 성과

## 5.Prompt template


### 1)Prompt template
- Prompt Template 은 LLM에게 보낼 프롬프트를 동적으로 생성하고 관리하기 위한 핵심 기능.
- 매번 프롬프트를 직접 작성하는 대신, 템플릿으로 만들어두면 변수만 바꿔가며 일관된 프롬프트를 재사용할 수 있습니다. LangChain의 PromptTemplate은 프롬프트를 코드처럼 관리하게 해주는 핵심 도구